# Importance of the Project
**The COVID-19 pandemic has caused significant disruption globally, and the situation continues to evolve. The analysis and prediction of COVID-19 spread are essential for effective public health policies and prevention strategies. This project aims to analyze COVID-19 data from various sources and develop models to predict the future spread of the virus.**

### Dataset Description
Dataset contails symptoms of patients which is crucial to identify the infection of covid. Columns are categorical in nature.
Details of the columns are :
*  ID (Individual ID)

* Sex (male/female). 

* Age ≥60 above years (true/false) 

* Test date (date when tested for COVID)

* Cough (true/false).

* Fever (true/false). 

* Sore throat (true/false). 
* Shortness of breath (true/false). 

* Headache (true/false). 
* Known contact with an individual confirmed to have COVID-19 (true/false).
* Corona positive or negative


In [ ]:
# importing necessary libraries
import numpy
import pandas
import seaborn
import matplotlib.pyplot as plt

from sklearn.impute import KNNImputer

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, make_scorer, recall_score,precision_score,f1_score,roc_curve, auc, confusion_matrix


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# loading the dataset
covid = pandas.read_csv('/kaggle/input/corona-tested-dataset/corona_tested_006.csv')

In [ ]:
covid.head()   # first 5 rows

In [ ]:
covid.tail()     # last 5 rows

**covid symtoms have 'true' and 'True'. In english meaning is same but python interprets as different**

In [ ]:
# defining a function to convert the data
def convert(x):
    if x=='TRUE':
        return 'true'
    elif x==True:
        return 'true'
    elif x=='FALSE' or x==False:
        return 'false'
    else:
        return x

In [ ]:
# mapping the function

covid['Cough_symptoms'] = covid['Cough_symptoms'].map(convert)
covid['Fever'] = covid['Fever'].map(convert)
covid['Sore_throat'] = covid['Sore_throat'].map(convert)
covid['Shortness_of_breath'] = covid['Shortness_of_breath'].map(convert)
covid['Headache'] = covid['Headache'].map(convert)

In [ ]:
# renaming a column
covid.rename(columns={'Sex':'Gender'},inplace=True)

In [ ]:
covid.info()

**In the dataset the value 'None' should be replaced with NaN**

In [ ]:
covid.replace({'None':numpy.nan},inplace=True)

In [ ]:
# Finding the total null values
covid.isnull().sum()

In [ ]:
for i in covid.columns:
    print(f"{i} has",round(covid[i].isnull().sum()*100/covid.shape[0],5),"%")

In [ ]:
# dropping nan rows for columns having less than 1% nan
covid.dropna(subset=['Cough_symptoms','Fever','Sore_throat','Shortness_of_breath','Headache'],axis=0,inplace=True)

In [ ]:
# drop age_60_above column 
covid.drop('Age_60_above',axis=1,inplace=True)

In [ ]:
covid.replace({'other':numpy.nan},inplace=True)

In [ ]:

covid.dropna(subset=['Corona'],axis=0,inplace=True)

In [ ]:
# converting the datatype as categorical
for i in covid.columns:
    if i=='Ind_ID' or i == 'Test_date' or i == 'Test_date':
        pass
    else:
        covid[i] = covid[i].astype('category')

In [ ]:
covid.info()

In [ ]:
#univatiate analysis

plt.figure(figsize=(4,3))
seaborn.countplot(x='Cough_symptoms',hue='Corona',data=covid,width=0.4)
plt.show()


In [ ]:
plt.figure(figsize=(4,3))
seaborn.countplot(x='Fever',hue='Corona',data=covid,width=0.4)
plt.show()


In [ ]:
plt.figure(figsize=(4,3))
seaborn.countplot(x='Sore_throat',hue='Corona',data=covid,width=0.4)
plt.show()


In [ ]:
plt.figure(figsize=(4,3))
seaborn.countplot(x='Shortness_of_breath',hue='Corona',data=covid,width=0.4)
plt.show()


In [ ]:
plt.figure(figsize=(4,3))
seaborn.countplot(x='Headache',hue='Corona',data=covid,width=0.4)
plt.show()


In [ ]:
plt.figure(figsize=(4,3))
seaborn.countplot(x='Gender',hue='Corona',data=covid,width=0.4)
plt.show()


In [ ]:
plt.figure(figsize=(4,3))
seaborn.countplot(x='Known_contact',hue='Corona',data=covid,width=0.4)
plt.show()


In [ ]:
covid.isnull().sum()

In [ ]:
covid_data=covid.copy(deep=True)

## Encoding

In [ ]:
# Encoding
covid_data['Cough_symptoms']=pandas.get_dummies(covid_data['Cough_symptoms'],drop_first=True)
covid_data['Fever']=pandas.get_dummies(covid_data['Fever'],drop_first=True)
covid_data['Sore_throat']=pandas.get_dummies(covid_data['Sore_throat'],drop_first=True)
covid_data['Shortness_of_breath']=pandas.get_dummies(covid_data['Shortness_of_breath'],drop_first=True)
covid_data['Headache']=pandas.get_dummies(covid_data['Headache'],drop_first=True)
covid_data['Corona']=pandas.get_dummies(covid_data['Corona'],drop_first=True)

In [ ]:
covid_data['Gender'].replace({'male':0,'female':1},inplace=True)
covid_data['Known_contact'].replace({'Other':2,'Contact with confirmed':1,'Abroad':0},inplace=True)

In [ ]:
covid_data.isnull().sum()

## Imputation

In [ ]:
imputed_data = covid_data.copy(deep=True)

In [ ]:
# applying KNN imputation
knn=KNNImputer(n_neighbors=5,weights='uniform')
columns=['Cough_symptoms', 'Fever', 'Sore_throat','Shortness_of_breath', 'Headache','Gender','Known_contact']
imputed=knn.fit_transform(covid_data[columns])

In [ ]:
df=pandas.DataFrame(imputed,columns=columns)

In [ ]:
def change(x):
    if x>0.5:
        return 1
    elif x<0.5:
        return 0
    else:
        return x

In [ ]:
df['Gender'] = df['Gender'].apply(change)

In [ ]:
imputed_data['Gender'].iloc[:] =df['Gender']

In [ ]:
imputed_data.head()

In [ ]:
imputed_data.info()

In [ ]:
imputed_data['Gender'] = imputed_data['Gender'].astype('uint8') 

In [ ]:
final_data = imputed_data.astype('category')

In [ ]:
final_data.info()

# Feature Engineering


In [ ]:
# separating features and target
features = final_data.drop(['Ind_ID','Test_date','Corona'],axis=1)
target = final_data['Corona']

In [ ]:
# chi2 method to select important k best features

selector = SelectKBest(score_func=chi2, k=6)
X_new = selector.fit_transform(features, target)

idxs_selected = selector.get_support(indices=True)

feat_names = features.columns[idxs_selected]

print(feat_names)

**covid symptoms are the important features of the dataset**

In [ ]:
# splitting train test set
x_train,x_test,y_train,y_test = train_test_split(features,target,test_size=0.3,random_state=42)

In [ ]:
y_test.value_counts()

In [ ]:
y_train.value_counts()

**1. random forest classifier**

In [ ]:
rf=RandomForestClassifier()
rf.fit(x_train,y_train)

In [ ]:
pred_rf = rf.predict(x_test)

# accuracy score
accuracy_rf = accuracy_score(y_test, pred_rf)
print("Accuracy:", accuracy_rf*100)

##### Parametre tuning for random forest

In [ ]:
rfc = RandomForestClassifier(random_state=42)

param_grid = {'n_estimators': [50, 100, 200],
              'max_depth': [5, 10, 15, 20, None],
              'max_features': ['sqrt', 'log2'],
              'bootstrap': [True, False]}


scorer = make_scorer(accuracy_score)

grid_obj = GridSearchCV(rfc, param_grid, scoring=scorer)
grid_fit = grid_obj.fit(x_train, y_train)

In [ ]:
# Get the best hyperparameters
best_params = grid_fit.best_params_
best_params

In [ ]:
# Training the model using the best hyperparameters
rfc_best = RandomForestClassifier(random_state=42, **best_params)
rfc_best.fit(x_train, y_train)
y_pred_rf = rfc_best.predict(x_test)

In [ ]:
# Evaluate the best model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Accuracy: {:.10f}%".format(accuracy_rf * 100.0))


##### performance measures for random forest


In [ ]:
# confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix:\n", cm_rf)

# precision
prec_rf = cm_rf[0][0]*100/(cm_rf[0][0] + cm_rf[0][1])
print("Precision:", prec_rf)

# recall
recall_rf = cm_rf[0][0]*100/(cm_rf[0][0] + cm_rf[1][0])
print('recall :',recall_rf)

# F-1 Score
f1_rf=2*prec_rf*recall_rf/(recall_rf + prec_rf)
print("F1 Score:", f1_rf)

# False Negative Score
fnr_rf = cm_rf[1][0]*100/(cm_rf[0][0] + cm_rf[1][0])
print('False Negative rate : ',fnr_rf)

In [ ]:
# AUC ROC curve
fpr, tpr, thresholds = roc_curve(y_test, pred_rf)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


**2. Decision Tree Classifier**

In [ ]:
dt = DecisionTreeClassifier()

dt.fit(x_train, y_train)

In [ ]:
y_pred = dt.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy*100)

**Parameter tuning for Decision tree model**

In [ ]:
dtc = DecisionTreeClassifier(random_state=42)

param_grid = {
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

scorer = make_scorer(accuracy_score)

grid = GridSearchCV(dtc, param_grid=param_grid, scoring=scorer, cv=5)
grid.fit(x_train, y_train)

print('Best parameters:', grid.best_params_)

In [ ]:
dtc_best = DecisionTreeClassifier(max_depth = 7, min_samples_leaf = 4, min_samples_split = 10)
dtc_best.fit(x_train,y_train)

In [ ]:
y_dt = dtc_best.predict(x_test)

accuracy_dtc = accuracy_score(y_test, y_dt)
print('Test accuracy:', accuracy_dtc*100)


**Performance measures for Decision Tree Classifier**

In [ ]:
# confusion matrix
cm_dt = confusion_matrix(y_test, y_dt)
print("Confusion Matrix:\n", cm_dt)

# precision
prec_dt = cm_dt[0][0]*100/(cm_dt[0][0] + cm_dt[0][1])
print("Precision:", prec_dt)

# recall
recall_dt = cm_dt[0][0]*100/(cm_dt[0][0] + cm_dt[1][0])
print('recall :',recall_dt)

# F-1 Score
f1_dt=2*prec_dt*recall_dt/(recall_dt + prec_dt)
print("F1 Score:", f1_dt)

# False Negative Score
fnr_dt = cm_dt[1][0]*100/(cm_dt[0][0] + cm_dt[1][0])
print('False Negative rate : ',fnr_dt)

In [ ]:
# AUC ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_dt)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


**3. Naive Bias Classifier**

In [ ]:

nb = GaussianNB()

nb.fit(x_train, y_train)

y_pred_nb = nb.predict(x_test)


In [ ]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)
accuracy_nb

**As naive bias model giving poor accuracy so tuning may not be required**

**4. Adaboost classifier**

In [ ]:
adaboost = AdaBoostClassifier()
adaboost.fit(x_train,y_train)
y_pred_ada=adaboost.predict(x_test)
accuracy_ada =  accuracy_score(y_test, y_pred_ada)
accuracy_ada

**Parameter tuning for adaboost model**

In [ ]:
adaboost_params = {'n_estimators': [50, 100, 200],
                   'learning_rate': [0.1, 0.01, 0.001]}

adaboost_grid = GridSearchCV(adaboost, adaboost_params, cv=5)
adaboost_grid.fit(x_train, y_train)

print("Best Hyperparameters for AdaBoost: ", adaboost_grid.best_params_)

In [ ]:
adaboost = AdaBoostClassifier(learning_rate= 0.1, n_estimators=200)
adaboost.fit(x_train,y_train)
y_pred_ada=adaboost.predict(x_test)
accuracy_ada =  accuracy_score(y_test, y_pred_ada)
accuracy_ada

**Performance measures of Adaboost model**

In [ ]:
# confusion matrix
cm_ada = confusion_matrix(y_test, y_pred_ada)
print("Confusion Matrix:\n", cm_ada)

# precision
prec_ada = cm_ada[0][0]*100/(cm_ada[0][0] + cm_ada[0][1])
print("Precision:", prec_ada)

# recall
recall_ada = cm_ada[0][0]*100/(cm_ada[0][0] + cm_ada[1][0])
print('recall :',recall_rf)

# F-1 Score
f1_ada=2*prec_ada*recall_ada/(recall_ada + prec_ada)
print("F1 Score:", f1_rf)

# False Negative Score
fnr_ada = cm_ada[1][0]*100/(cm_ada[0][0] + cm_ada[1][0])
print('False Negative rate : ',fnr_ada)

In [ ]:
# AUC ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_ada)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


**5. Support Vector Machine Classifier**

In [ ]:
svc = SVC()
svc.fit(x_train,y_train)

In [ ]:
pred_svc=svc.predict(x_test)
acc_svc=accuracy_score(pred_svc,y_test)
acc_svc*100

In [ ]:
# confusion matrix
cm_svc = confusion_matrix(y_test, pred_svc)
print("Confusion Matrix:\n", cm_svc)

# precision
prec_svc = cm_svc[0][0]*100/(cm_svc[0][0] + cm_svc[0][1])
print("Precision:", prec_svc)

# recall
recall_svc = cm_svc[0][0]*100/(cm_svc[0][0] + cm_svc[1][0])
print('recall :',recall_svc)

# F-1 Score
f1_svc = 2*prec_svc*recall_svc/(recall_svc + prec_svc)
print("F1 Score:", f1_svc)

# False Negative Score
fnr_svc = cm_svc[1][0]*100/(cm_svc[0][0] + cm_svc[1][0])
print('False Negative rate : ',fnr_svc)

In [ ]:
# AUC ROC curve
fpr, tpr, thresholds = roc_curve(y_test, pred_svc)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


In [ ]:
df = pandas.DataFrame({'random forest':[accuracy_rf*100,prec_rf,recall_rf,f1_rf,fnr_rf,0.79],
                      'naive bayes':[accuracy_nb*100,'nan','nan','nan','nan','nan'],
                      'Adaboost':[accuracy_ada*100,prec_ada,recall_ada,f1_ada,fnr_ada,0.67],
                      'Decision Tree':[accuracy_dtc*100,prec_dt,recall_dt,f1_dt,fnr_dt,0.77],
                       'SVC':[acc_svc*100,prec_svc,recall_svc,f1_svc,fnr_svc,0.77]
                      },index=['Accuracy','Precision','Recall','F-1 Score','False Negative Rate', 'ROC curve area'])
df

**Random forest model we will prefer in this case as FNR is lowe as well as ROC area is more.**